In [1]:
import torch

print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("current device:", torch.cuda.current_device())
    print("device name:", torch.cuda.get_device_name(0))


cuda available: True
device count: 1
current device: 0
device name: NVIDIA A100-SXM4-40GB


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!cp -r "/content/drive/MyDrive/Kaggle_Data" /content/Kaggle_Data

In [4]:
!ls /content
!ls /content/Kaggle_Data

drive  Kaggle_Data  sample_data
audio  metadata  outputs


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet50
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from torch.optim import AdamW
from tqdm import tqdm
from pathlib import Path
import shutil

# ====================================================
# CONFIGURATION
# ====================================================
class Config:
    seed = 42

    # Audio / mel-spectrogram
    sample_rate = 22050
    duration = 4.0
    n_mels = 128
    n_fft = 2048
    hop_length = 512
    fmin = 0
    fmax = 8000

    # Training hyperparameters
    batch_size = 48
    n_folds = 5
    epochs = 30
    learning_rate = 7e-4
    weight_decay = 1e-4
    label_smoothing = 0.05
    mixup_alpha = 0.3
    grad_clip = 1.0

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    use_cache = True

    # --------- Paths ---------
    base_dir = Path("/content/drive/MyDrive/Kaggle_Data")

    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        hits = list(kaggle_root.rglob("kaggle_train.csv"))
        if len(hits) > 0:
            base_dir = hits[0].parents[1]

    train_csv = base_dir / "metadata" / "kaggle_train.csv"
    test_csv  = base_dir / "metadata" / "kaggle_test.csv"
    audio_dir = base_dir / "audio"

    if Path("/kaggle/working").exists():
        cache_dir = Path("/kaggle/working/mel_cache")
        output_dir = Path("/kaggle/working/models_resnet_audio")
    else:
        cache_dir = Path("./mel_cache")
        output_dir = Path("./models_resnet_audio")

    # DataLoader
    num_workers = max(2, min(8, os.cpu_count() or 2))
    prefetch_factor = 2

Config.output_dir.mkdir(parents=True, exist_ok=True)
Config.cache_dir.mkdir(parents=True, exist_ok=True)

# ====================================================
# UTILS & REPRODUCIBILITY
# ====================================================
def seed_everything(seed: int):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(Config.seed)

# ====================================================
# DATASET
# ====================================================
class AudioDataset(Dataset):
    """
    Output: 3-channel spectrogram image -> resize (3,224,224)
    Cache stores RAW standardized 3ch feature
    """
    def __init__(self, df, audio_dir, phase="train"):
        self.df = df.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.phase = phase
        self.target_length = int(Config.sample_rate * Config.duration)

    def __len__(self):
        return len(self.df)

    def _load_audio(self, file_path: Path) -> np.ndarray:
        try:
            y, _ = librosa.load(str(file_path), sr=Config.sample_rate)
        except Exception as e:
            print(f"Error loading {file_path}: {e}")
            y = np.zeros(self.target_length, dtype=np.float32)

        if len(y) > self.target_length:
            y = y[:self.target_length]
        else:
            y = np.pad(y, (0, self.target_length - len(y)), mode="constant")
        return y.astype(np.float32)

    def _compute_3ch_mel(self, y: np.ndarray) -> np.ndarray:
        melspec = librosa.feature.melspectrogram(
            y=y,
            sr=Config.sample_rate,
            n_mels=Config.n_mels,
            n_fft=Config.n_fft,
            hop_length=Config.hop_length,
            fmin=Config.fmin,
            fmax=Config.fmax,
            power=2.0,
        )
        logmel = librosa.power_to_db(melspec, ref=np.max).astype(np.float32)

        delta = librosa.feature.delta(logmel).astype(np.float32)
        delta2 = librosa.feature.delta(logmel, order=2).astype(np.float32)

        x = np.stack([logmel, delta, delta2], axis=0)  # (3, F, T)

        # per-channel standardization
        mean = x.mean(axis=(1, 2), keepdims=True)
        std = x.std(axis=(1, 2), keepdims=True) + 1e-6
        x = (x - mean) / std
        return x

    # --------- augmentations ---------
    def _spec_light_aug(self, spec: torch.Tensor) -> torch.Tensor:
        if self.phase != "train":
            return spec

        # random gain
        if random.random() < 0.5:
            gain = 10 ** (random.uniform(-6, 6) / 20)
            spec = spec * gain

        # random gaussian noise
        if random.random() < 0.3:
            spec = spec + torch.randn_like(spec) * random.uniform(0.01, 0.05)

        # random time shift
        if random.random() < 0.3:
            shift = random.randint(-20, 20)
            spec = torch.roll(spec, shifts=shift, dims=2)

        return spec

    def apply_spec_augment(self, spec: torch.Tensor,
                           num_masks=2, freq_mask_param=24, time_mask_param=48):
        if self.phase != "train":
            return spec

        spec = spec.clone()
        C, Fdim, Tdim = spec.shape

        for _ in range(num_masks):
            f = random.randint(0, freq_mask_param)
            if f > 0:
                f0 = random.randint(0, max(0, Fdim - f))
                spec[:, f0:f0+f, :] = 0

            t = random.randint(0, time_mask_param)
            if t > 0:
                t0 = random.randint(0, max(0, Tdim - t))
                spec[:, :, t0:t0+t] = 0

        return spec

    def _atomic_save_npy(self, path: Path, arr: np.ndarray):
        tmp = path.with_suffix(".tmp.npy")
        np.save(tmp, arr)
        os.replace(tmp, path)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        if self.phase == "test":
            file_path = self.audio_dir / "test" / row["slice_file_name"]
            fold_tag = "test"
        else:
            file_path = self.audio_dir / f"fold{row['fold']}" / row["slice_file_name"]
            fold_tag = f"fold{row['fold']}"

        cache_key = f"{fold_tag}_{row['slice_file_name']}".replace("/", "_")
        cache_key = f"{cache_key}_sr{Config.sample_rate}_dur{Config.duration}_m{Config.n_mels}_hl{Config.hop_length}"
        cache_path = Config.cache_dir / (cache_key + ".npy")

        if Config.use_cache and cache_path.exists():
            x = np.load(cache_path)
        else:
            y = self._load_audio(file_path)
            x = self._compute_3ch_mel(y)
            if Config.use_cache:
                try:
                    self._atomic_save_npy(cache_path, x)
                except Exception:
                    pass

        image = torch.tensor(x, dtype=torch.float32)  # (3,F,T)
        image = F.interpolate(
            image.unsqueeze(0),
            size=(224, 224),
            mode="bilinear",
            align_corners=False
        ).squeeze(0)

        image = self._spec_light_aug(image)
        image = self.apply_spec_augment(image)

        label = row["classID"] if "classID" in row else -1
        return image, torch.tensor(label, dtype=torch.long)

# ====================================================
# MODEL (ResNet50)
# ====================================================
class AudioResNet(nn.Module):
    def __init__(self, num_classes=10, dropout=0.2):
        super().__init__()
        self.backbone = resnet50(weights=None)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, num_classes)
        )

        for m in self.backbone.fc.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.backbone(x)

# ====================================================
# TRAINING HELPERS
# ====================================================
def mixup_data(x, y, alpha, device):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    bs = x.size(0)
    index = torch.randperm(bs, device=device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def get_amp(device):
    if device.type != "cuda":
        class DummyScaler:
            def scale(self, x): return x
            def step(self, opt): opt.step()
            def update(self): pass
            def unscale_(self, opt): pass
        def autocast_ctx():
            return torch.autocast("cpu", enabled=False)
        return DummyScaler(), autocast_ctx

    # CUDA
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        scaler = torch.amp.GradScaler("cuda", enabled=True)
        def autocast_ctx():
            return torch.amp.autocast("cuda", dtype=torch.float16, enabled=True)
        return scaler, autocast_ctx
    else:
        scaler = torch.cuda.amp.GradScaler(enabled=True)
        def autocast_ctx():
            return torch.cuda.amp.autocast(enabled=True)
        return scaler, autocast_ctx

def train_one_epoch(model, loader, optimizer, scheduler, criterion, device, scaler, autocast_ctx):
    model.train()
    running_loss = 0.0

    pbar = tqdm(loader, desc="Training", leave=False)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        images, y_a, y_b, lam = mixup_data(images, labels, Config.mixup_alpha, device)

        optimizer.zero_grad(set_to_none=True)

        with autocast_ctx():
            outputs = model(images)
            loss = mixup_criterion(criterion, outputs, y_a, y_b, lam)

        # backward
        if device.type == "cuda" and hasattr(scaler, "scale"):
            scaler.scale(loss).backward()

            if Config.grad_clip is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), Config.grad_clip)

            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            if Config.grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), Config.grad_clip)
            optimizer.step()

        scheduler.step()

        running_loss += loss.item()
        pbar.set_postfix(loss=float(loss.item()), lr=float(scheduler.get_last_lr()[0]))

    return running_loss / max(1, len(loader))

@torch.no_grad()
def validate(model, loader, criterion, device, autocast_ctx):
    model.eval()
    running_loss = 0.0
    preds, targets = [], []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with autocast_ctx():
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item()
        preds.append(outputs.argmax(1).detach().cpu().numpy())
        targets.append(labels.detach().cpu().numpy())

    preds = np.concatenate(preds)
    targets = np.concatenate(targets)

    acc = accuracy_score(targets, preds)
    macro_f1 = f1_score(targets, preds, average="macro")
    score = 0.8 * acc + 0.2 * macro_f1
    return running_loss / max(1, len(loader)), acc, macro_f1, score

def make_loader(dataset, shuffle):
    pin = (Config.device.type == "cuda")
    nw = Config.num_workers

    kwargs = dict(
        batch_size=Config.batch_size,
        shuffle=shuffle,
        num_workers=nw,
        pin_memory=pin,
        drop_last=shuffle,
    )
    if nw > 0:
        kwargs["persistent_workers"] = True
        kwargs["prefetch_factor"] = Config.prefetch_factor

    return DataLoader(dataset, **kwargs)

# ====================================================
# MAIN
# ====================================================
def main():
    print(f"Using Device: {Config.device}")
    print(f"base_dir   : {Config.base_dir}")
    print(f"train_csv  : {Config.train_csv}")
    print(f"test_csv   : {Config.test_csv}")
    print(f"audio_dir  : {Config.audio_dir}")
    print(f"cache_dir  : {Config.cache_dir}")
    print(f"output_dir : {Config.output_dir}")
    print(f"num_workers= {Config.num_workers}")

    train_df = pd.read_csv(Config.train_csv)
    test_df  = pd.read_csv(Config.test_csv)

    skf = StratifiedKFold(n_splits=Config.n_folds, shuffle=True, random_state=Config.seed)

    oof_preds  = np.zeros((len(train_df), 10), dtype=np.float32)
    test_preds = np.zeros((len(test_df), 10), dtype=np.float32)

    # test loader once
    test_ds = AudioDataset(test_df, Config.audio_dir, phase="test")
    test_loader = make_loader(test_ds, shuffle=False)

    scaler, autocast_ctx = get_amp(Config.device)

    for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df["classID"])):
        print(f"\n{'='*20} Fold {fold+1}/{Config.n_folds} {'='*20}")

        train_fold = train_df.iloc[train_idx].reset_index(drop=True)
        val_fold   = train_df.iloc[val_idx].reset_index(drop=True)

        train_ds = AudioDataset(train_fold, Config.audio_dir, phase="train")
        val_ds   = AudioDataset(val_fold,   Config.audio_dir, phase="val")

        train_loader = make_loader(train_ds, shuffle=True)
        val_loader   = make_loader(val_ds,   shuffle=False)

        model = AudioResNet(num_classes=10, dropout=0.2).to(Config.device)

        optimizer = AdamW(model.parameters(), lr=Config.learning_rate, weight_decay=Config.weight_decay)

        # class weights for MacroF1
        counts = train_fold["classID"].value_counts().sort_index().values.astype(np.float32)
        class_w = 1.0 / np.sqrt(counts)
        class_w = torch.tensor(class_w, dtype=torch.float32, device=Config.device)

        criterion = nn.CrossEntropyLoss(weight=class_w, label_smoothing=Config.label_smoothing)

        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=Config.learning_rate,
            steps_per_epoch=len(train_loader),
            epochs=Config.epochs,
            pct_start=0.15,
            div_factor=25.0,
            final_div_factor=1e4
        )

        best_score = -1e9
        best_model_path = Config.output_dir / f"resnet50_fromscratch_fold{fold+1}.pth"

        for epoch in range(Config.epochs):
            train_loss = train_one_epoch(model, train_loader, optimizer, scheduler,
                                         criterion, Config.device, scaler, autocast_ctx)
            val_loss, val_acc, val_f1, val_score = validate(model, val_loader,
                                                            criterion, Config.device, autocast_ctx)

            print(
                f"Epoch {epoch+1:02d}: "
                f"Train Loss {train_loss:.4f} | "
                f"Val Loss {val_loss:.4f} | "
                f"Val Acc {val_acc:.4f} | "
                f"Val MacroF1 {val_f1:.4f} | "
                f"Val Score {val_score:.4f}"
            )

            # Save best score
            if val_score > best_score:
                best_score = val_score
                torch.save(model.state_dict(), best_model_path)
                print(f"  >>> Saved best! Score={best_score:.4f}")

        # Load best
        model.load_state_dict(torch.load(best_model_path, map_location=Config.device))
        model.eval()

        # OOF preds
        ptr = 0
        with torch.no_grad():
            for images, _ in val_loader:
                bs = images.size(0)
                images = images.to(Config.device, non_blocking=True)
                with autocast_ctx():
                    outputs = model(images)
                    probs = torch.softmax(outputs, dim=1).float().cpu().numpy()

                original_indices = val_idx[ptr:ptr+bs]
                oof_preds[original_indices] = probs
                ptr += bs

        # Test preds
        fold_test = []
        with torch.no_grad():
            for images, _ in tqdm(test_loader, desc=f"Fold {fold+1} Inference"):
                images = images.to(Config.device, non_blocking=True)
                with autocast_ctx():
                    outputs = model(images)
                    probs = torch.softmax(outputs, dim=1).float().cpu().numpy()
                fold_test.append(probs)

        test_preds += np.concatenate(fold_test) / Config.n_folds

    # Submission
    print(f"\n{'='*20} Generating Submission {'='*20}")
    final_predictions = np.argmax(test_preds, axis=1)
    submission = pd.DataFrame({"ID": test_df["ID"], "TARGET": final_predictions})
    submission_path = "submission.csv"
    submission.to_csv(submission_path, index=False)
    print(f"Submission saved to {submission_path}")

    # OOF metrics
    oof_labels = np.argmax(oof_preds, axis=1)
    oof_acc = accuracy_score(train_df["classID"], oof_labels)
    oof_f1  = f1_score(train_df["classID"], oof_labels, average="macro")
    oof_score = 0.8 * oof_acc + 0.2 * oof_f1
    print(f"Overall OOF Acc: {oof_acc:.4f} | MacroF1: {oof_f1:.4f} | Score: {oof_score:.4f}")

    return train_df, oof_preds, test_preds

if __name__ == "__main__":
    main()


Using Device: cuda
base_dir   : /content/drive/MyDrive/Kaggle_Data
train_csv  : /content/drive/MyDrive/Kaggle_Data/metadata/kaggle_train.csv
test_csv   : /content/drive/MyDrive/Kaggle_Data/metadata/kaggle_test.csv
audio_dir  : /content/drive/MyDrive/Kaggle_Data/audio
cache_dir  : mel_cache
output_dir : models_resnet_audio
num_workers= 8

==================== Fold 1/5 ====================


Training:   0%|          | 0/117 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/optim/lr_scheduler.py:192: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


Epoch 01: Train Loss 2.5184 | Val Loss 2.3770 | Val Acc 0.2542 | Val MacroF1 0.1701 | Val Score 0.2374
  >>> Saved best! Score=0.2374


Epoch 02: Train Loss 2.2840 | Val Loss 2.2327 | Val Acc 0.3821 | Val MacroF1 0.3399 | Val Score 0.3736
  >>> Saved best! Score=0.3736


Epoch 03: Train Loss 1.9520 | Val Loss 2.2574 | Val Acc 0.3284 | Val MacroF1 0.2719 | Val Score 0.3171


...

Epoch 28: Train Loss 0.7290 | Val Loss 0.4175 | Val Acc 0.9618 | Val MacroF1 0.9648 | Val Score 0.9624
  >>> Saved best! Score=0.9624


Epoch 29: Train Loss 0.7732 | Val Loss 0.4226 | Val Acc 0.9597 | Val MacroF1 0.9615 | Val Score 0.9601


Epoch 30: Train Loss 0.8709 | Val Loss 0.4226 | Val Acc 0.9604 | Val MacroF1 0.9630 | Val Score 0.9609


Fold 5 Inference: 100%|██████████| 35/35 [00:00<00:00, 46.53it/s]



==================== Generating Submission ====================
Submission saved to submission.csv
Overall OOF Acc: 0.9682 | MacroF1: 0.9699 | Score: 0.9686
